In [2]:
import os
import sys
from pathlib import Path

In [5]:
import sys
import subprocess

# Install required packages from the notebook using the current Python executable
required = [
    "langchain",
    "langchain-core",
    "langchain-text-splitters",
    "langchain-ollama",
    "langchain-community",
    "faiss-cpu",
    "rank-bm25",
    "pypdf",
]

def install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade'] + packages
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
    return result.returncode

# Attempt install
rc = install(required)
if rc == 0:
    print('Install completed successfully.')
else:
    print('One or more packages failed to install. Check errors above.')

Running: c:\Users\arpit\anaconda3\python.exe -m pip install --upgrade langchain langchain-core langchain-text-splitters langchain-ollama langchain-community faiss-cpu rank-bm25 pypdf
  Attempting uninstall: websockets
    Found existing installation: websockets 16.0
    Uninstalling websockets-16.0:
      Successfully uninstalled websockets-16.0

Install completed successfully.
  Attempting uninstall: websockets
    Found existing installation: websockets 16.0
    Uninstalling websockets-16.0:
      Successfully uninstalled websockets-16.0

Install completed successfully.


In [8]:
import sys, traceback

print("Executing with Python Version:", sys.version)
print("Verifying library imports...")

checks = [
    ("langchain_community.document_loaders.PyPDFLoader", "from langchain_community.document_loaders import PyPDFLoader"),
    ("langchain_text_splitters.RecursiveCharacterTextSplitter", "from langchain_text_splitters import RecursiveCharacterTextSplitter"),
    ("langchain_community.vectorstores.FAISS", "from langchain_community.vectorstores import FAISS"),
    ("langchain_ollama.OllamaEmbeddings, ChatOllama", "from langchain_ollama import OllamaEmbeddings, ChatOllama"),
    ("langchain_community.retrievers.BM25Retriever", "from langchain_community.retrievers import BM25Retriever"),
    ("langchain.retrievers.EnsembleRetriever", "from langchain.retrievers import EnsembleRetriever"),
    ("langchain_core.prompts.ChatPromptTemplate", "from langchain_core.prompts import ChatPromptTemplate"),
    ("langchain_core.runnables.RunnablePassthrough", "from langchain_core.runnables import RunnablePassthrough"),
    ("langchain_core.output_parsers.StrOutputParser", "from langchain_core.output_parsers import StrOutputParser"),
]

missing = []
for name, stmt in checks:
    try:
        exec(stmt, globals())
        print(f"OK: {name}")
    except Exception as e:
        tb = traceback.format_exc()
        missing.append((name, str(e), tb))
        print(f"MISSING: {name} -> {e}")

if missing:
    print('\nSummary of missing imports:')
    for name, err, tb in missing:
        print(f'- {name}: {err}')
    print('\nNotes:')
    print('- `FAISS` (faiss-cpu) is not available via pip on some Windows/python combos. Use conda: `conda install -c conda-forge faiss-cpu` or run on Linux.')
    print('- If `langchain.retrievers` or other langchain submodules are missing, try restarting the kernel and ensuring the notebook is using the same Python interpreter used for pip installs (sys.executable).')
    print('- To force-reinstall langchain:')
    print(f'    {sys.executable} -m pip install --upgrade langchain langchain-core langchain-community')
else:
    print('\nAll imports available.')


Executing with Python Version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Verifying library imports...
OK: langchain_community.document_loaders.PyPDFLoader
OK: langchain_text_splitters.RecursiveCharacterTextSplitter
OK: langchain_community.vectorstores.FAISS
OK: langchain_ollama.OllamaEmbeddings, ChatOllama
OK: langchain_community.retrievers.BM25Retriever
MISSING: langchain.retrievers.EnsembleRetriever -> No module named 'langchain.retrievers'
OK: langchain_core.prompts.ChatPromptTemplate
OK: langchain_core.runnables.RunnablePassthrough
OK: langchain_core.output_parsers.StrOutputParser

Summary of missing imports:
- langchain.retrievers.EnsembleRetriever: No module named 'langchain.retrievers'

Notes:
- `FAISS` (faiss-cpu) is not available via pip on some Windows/python combos. Use conda: `conda install -c conda-forge faiss-cpu` or run on Linux.
- If `langchain.retrievers` or other langchain submodules are missing, try restarting the

In [9]:
# Retriever helper: prefer EnsembleRetriever class if available, otherwise fall back to BM25Retriever class
try:
    from langchain.retrievers import EnsembleRetriever
    retriever_class = EnsembleRetriever
    print("EnsembleRetriever is available (class).")
except Exception:
    from langchain_community.retrievers import BM25Retriever
    retriever_class = BM25Retriever
    print("EnsembleRetriever not available — using BM25Retriever as fallback (class).")

# Expose a simple accessor: users should instantiate with the appropriate required args (docs, etc.)
def get_retriever_class():
    """Return the retriever class to instantiate later with proper arguments."""
    return retriever_class

# Example: show the class but do not instantiate (avoids validation errors)
print("Retriever class:", retriever_class)

EnsembleRetriever not available — using BM25Retriever as fallback (class).
Retriever class: <class 'langchain_community.retrievers.bm25.BM25Retriever'>


In [ ]:
# Jupyter Notebook - Cell 2: Document Ingestion and Layout Extraction
DATA_DIR = Path("books/pdf")
DATA_DIR.mkdir(exist_ok=True)

def ingest_ncert_pdfs(data_path: Path):
    """
    Scans the data directory, extracts text from PDF textbooks page-by-page,
    and returns a structured list of LangChain Document objects.
    """
    pdf_paths = list(data_path.glob("*.pdf"))
    if not pdf_paths:
        print(f"Warning: No PDF files found in {data_path.absolute()}. Please upload textbooks.")
        return
    
    parsed_pages = []
    for pdf_file in pdf_paths:
        print(f"Ingesting: {pdf_file.name}")
        # PyPDFLoader parses page-by-page to preserve structure and coordinate page-level citations
        loader = PyPDFLoader(str(pdf_file))
        try:
            pages = loader.load()
            parsed_pages.extend(pages)
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
        
    print(f"Successfully processed {len(parsed_pages)} total pages.")
    return parsed_pages

# Execute the ingestion pipeline
raw_pages = ingest_ncert_pdfs(DATA_DIR)

<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:2: SyntaxWarning: invalid escape sequence '\p'
C:\Users\arpit\AppData\Local\Temp\ipykernel_38936\268912381.py:2: SyntaxWarning: invalid escape sequence '\p'
  DATA_DIR = Path("books\pdf")


Ingesting: jeff101.pdf
Ingesting: jeff102.pdf
Ingesting: jeff103.pdf
Ingesting: jeff104.pdf
Ingesting: jeff105.pdf
Ingesting: jeff106.pdf
Ingesting: jeff107.pdf
Ingesting: jeff108.pdf
Ingesting: jeff109.pdf
Ingesting: jeff1ps.pdf
Ingesting: jefp101.pdf
Ingesting: jefp102.pdf
Ingesting: jefp103.pdf
Ingesting: jefp104.pdf
Ingesting: jefp105.pdf
